# When Does Feedback Help?

### Planning and model mismatch in Hessian-free Newton control

This notebook is a **thin layer over the package**. It imports the same functions the
scripts and the manuscript use; it does not reimplement any optimizer or statistic. If a
number here disagrees with the paper, the notebook is wrong, not the paper.

Three levels of reproduction exist in this repository. This notebook covers the first
two.

| Level | What it does | Cost |
|---|---|---|
| **Quick** | recompute every reported statistic and regenerate the figures from `results/public/` | a few minutes |
| **Smoke** | run one small optimization end to end, so you can see the machinery work | seconds |
| **Full** | re-run all experiments from scratch | long; see `docs/reproduce.md` |

The full planner suite is **not** run here. It is an oracle diagnostic that consumes
roughly a thousand times the deployment budget in search.

## 0. Setup

In [ ]:
import subprocess
import sys
from pathlib import Path

# 이 저장소의 src 를 우선한다. 설치 없이 clone 만 한 상태에서도 돌게 한다.
ROOT = Path.cwd() if (Path.cwd() / "results" / "public").is_dir() else Path.cwd().parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PUBLIC = ROOT / "results" / "public"
assert PUBLIC.is_dir(), f"results/public not found under {ROOT}"

# sys.path 를 먼저 세운 뒤 import 해야 하므로 셀 상단이 아니다.
from rl_newton.benchmark.metrics import median_of  # noqa: E402
from rl_newton.reporting import (  # noqa: E402
    load_public_results,
    paired,
    positive_count,
    public_roles,
)

print("repository:", ROOT)
print("published result files:")
for path in sorted(PUBLIC.glob("*")):
    print("   ", path.name)

## 1. The question

Truncated-Newton performance depends on two computational-resource decisions: the
damping level and the per-step CG iteration budget. Adapting them during optimization
looks like a reinforcement learning problem. Before training a policy, ask where the
benefit would come from.

- **Q1** — does a good action *sequence* exist?
- **Q2** — is it worth *revising* that sequence with feedback during execution?

Only if **Q2** holds is a per-step feedback policy justified. To separate them we compare
a ladder of controllers at an identical budget of 150 gradient-equivalent units (GE).

In [ ]:
LADDER = [
    ("best_static", "constant action, best of six candidates tuned on dev seeds"),
    ("best_open_loop", "four-segment schedule on a resource clock; state-blind"),
    ("heuristic", "rule-based damping adjustment"),
    ("onestep_narrow", "enumerate each step, take the immediately most efficient action"),
    ("committed_Q4_narrow", "plan once at the initial state, then execute it unchanged"),
    ("shrinking_Q4_narrow", "replan at every step with the remaining quota"),
]
width = max(len(name) for name, _ in LADDER)
for name, note in LADDER:
    print(f"{name:<{width}}  {note}")

print()
print("committed vs shrinking is the comparison that isolates feedback:")
print("  both receive the same budget and see the same instances,")
print("  and only shrinking observes the state during execution.")

## 2. Recompute the predeclared gates

The held-out confirmation covers 4 ill-conditioned SPD quadratic specs
(`d = 100`, `κ ∈ {1e3, 1e4, 1e5, 1e6}`) × 10 held-out seeds = 40 instances.

`best_static` and `best_open_loop` are not controllers but **tuning outcomes**, and the
selected candidate differs between experiments. The published CSV carries a
`controller_role` column so this is unambiguous.

In [ ]:
runs = load_public_results(PUBLIC / "heldout_quadratic.csv")
roles = public_roles(PUBLIC / "heldout_quadratic.csv")

print(f"{len(runs)} runs")
print("selected baselines:", roles)

GATES = [
    ("A2", "best_static", "shrinking_Q4_narrow", "replanning vs tuned constant"),
    ("C2", "onestep_narrow", "shrinking_Q4_narrow", "multi-step planning vs one-step"),
    ("C3", "committed_Q4_narrow", "shrinking_Q4_narrow", "feedback vs committed plan"),
]

print()
print(f"{'gate':<5} {'median':>8} {'95% CI':>22} {'p':>10} {'positive':>9}  comparison")
for gate, base, treat, note in GATES:
    d = paired(runs, base, treat, roles=roles)
    pos, n = positive_count(runs, base, treat, roles=roles)
    p = "<0.0001" if d.p_value < 1e-4 else f"{d.p_value:.4f}"
    ci = f"[{d.delta_ci[0]:+.3f}, {d.delta_ci[1]:+.3f}]"
    print(f"{gate:<5} {d.median_delta:>+8.3f} {ci:>22} {p:>10} {pos:>5}/{n:<3}  {note}")

Those three lines are the paper's core result.

- **Q1 is supported.** A good multi-step sequence exists, and `C2` confirms that
  multi-step lookahead beats one-step greedy control.
- **Q2 is not supported under these conditions.** `C3` is small, its interval is narrow
  and includes zero, and we did not observe a practically large feedback benefit.

We do **not** claim the feedback effect is zero. No equivalence margin was preregistered,
so "narrow interval containing zero" is all the data supports.

On a deterministic objective the planner's internal model is exact, so the plan formed at
the initial state is already the optimal prediction and replanning gains no new
information. Section 6 tests what happens when that assumption breaks.

## 3. Where the improvement comes from

**The median of paired differences is not linear.** The two blocks below are separate
statistics. Do not subtract entries across them — the paper contains a correction for
exactly that mistake.

In [ ]:
print("relative to the tuned constant setting")
for treat in ("best_open_loop", "onestep_narrow", "committed_Q4_narrow", "shrinking_Q4_narrow"):
    d = paired(runs, "best_static", treat, roles=roles)
    pos, n = positive_count(runs, "best_static", treat, roles=roles)
    print(f"  {treat:<22} {d.median_delta:>+7.3f}   {pos}/{n}")

print()
print("directly measured increments")
for gate, base, treat, _ in GATES[1:]:
    d = paired(runs, base, treat, roles=roles)
    print(f"  {gate}  {treat} - {base:<22} {d.median_delta:>+7.3f}")

print()
print("committed scores higher than replanning against the constant baseline,")
print("yet the direct replanning - committed difference is near zero.")
print("Both are true: the pooled medians fall on different instances.")
print("Per spec the two planners are effectively tied:")
for spec in sorted({r.task_instance_id.rsplit("_seed", 1)[0] for r in runs}):
    d = paired(runs, "committed_Q4_narrow", "shrinking_Q4_narrow", spec=spec, roles=roles)
    print(f"  {spec:<34} {d.median_delta:>+7.3f}")

## 4. What the planner costs

This is the practical caveat. The planner is a measurement instrument for how much
headroom exists, not an optimizer you would deploy.

In [ ]:
BUDGET_GE = 150.0
by_controller = {}
for run in runs:
    by_controller.setdefault(run.controller, []).append(run)

print(f"{'controller':<22} {'search GE':>10} {'x budget':>10}")
for name, _ in LADDER:
    label = roles.get(name, name)
    sub = by_controller.get(label)
    if not sub:
        continue
    ge = median_of([r.search_cost_ge for r in sub])
    ratio = "--" if ge <= 0 else f"{ge / BUDGET_GE:,.1f}x"
    print(f"{name:<22} {ge:>10,.0f} {ratio:>10}")

print()
print("Search GE measures simulated oracle work, not wall-clock compute.")

## 5. Conditioning (exploratory)

The specs were not designed as a factorial study of the condition number, so this is
descriptive. Note that the effect does **not** grow with `κ`; the largest value is at the
smallest condition number. We claim no monotone relation in either direction.

In [ ]:
print(f"{'spec':<34} {'A2 median':>10} {'positive':>9}")
for spec in sorted({r.task_instance_id.rsplit("_seed", 1)[0] for r in runs}):
    d = paired(runs, "best_static", "shrinking_Q4_narrow", spec=spec, roles=roles)
    pos, n = positive_count(
        runs, "best_static", "shrinking_Q4_narrow", spec=spec, roles=roles
    )
    print(f"{spec:<34} {d.median_delta:>+10.3f} {pos:>5}/{n:<3}")

## 6. Model mismatch (exploratory, `n = 3` per regime)

Here we changed **only the samples the optimizer observes**, for the same model and the
same data. `full-batch` is deterministic; the minibatch regimes are not.

`n = 3` per regime. We report signs and directions, **not magnitudes**, and we do not
quote confidence intervals or `p`-values for these runs.

In [ ]:
micro = PUBLIC / "micro_neural.csv"
mruns = load_public_results(micro, acceptance_rule="control")
mroles = public_roles(micro, acceptance_rule="control")

REGIMES = [
    ("mlp_d32_h128_c5_n512_fb", "R1 full-batch"),
    ("mlp_d32_h128_c5_n512_cs128", "R2 batch 128"),
    ("mlp_d32_h128_c5_n512_cs64", "R2 batch 64"),
]

print(f"{'regime':<16} {'C3 feedback':>12} {'C2 vs onestep':>14} {'A2 vs constant':>15}")
for spec, label in REGIMES:
    c3 = paired(mruns, "committed_Q4_narrow", "shrinking_Q4_narrow", spec=spec, roles=mroles)
    c2 = paired(mruns, "onestep_narrow", "shrinking_Q4_narrow", spec=spec, roles=mroles)
    a2 = paired(mruns, "best_static", "shrinking_Q4_narrow", spec=spec, roles=mroles)
    print(
        f"{label:<16} {c3.median_delta:>+12.3f} {c2.median_delta:>+14.3f} "
        f"{a2.median_delta:>+15.3f}"
    )

print()
print("why C3 turns positive under minibatch noise: the committed plan goes stale")
print(f"{'regime':<16} {'committed reject':>17} {'committed J_E':>14}")
for spec, label in REGIMES:
    sub = [
        r
        for r in mruns
        if r.controller == "committed_Q4_narrow"
        and r.task_instance_id.rsplit("_seed", 1)[0] == spec
    ]
    rate = median_of([r.rejection_rate for r in sub])
    j_e = median_of([r.log_improvement for r in sub])
    print(f"{label:<16} {rate:>17.2f} {j_e:>14.3f}")

Read that carefully. `C3 > 0` in the minibatch regimes does **not** mean replanning
became good. It means holding a stale plan became costly: the committed plan's rejection
rate rises and its terminal improvement collapses.

At the same time `C2` and `A2` go negative there — the planner falls below both the
inexpensive one-step controller and the tuned constant. So the positive
`committed → replanning` value is stale-plan avoidance, not an advantage over cheap
myopic control. That is the reason the study did not proceed to policy learning.

## 7. A small end-to-end run

Everything above reads published results. This cell actually optimizes something, so you
can see the mechanism. It is a 50-dimensional quadratic at the same 150 GE budget and
takes seconds.

In [ ]:
import math

from rl_newton.benchmark.paired import make_task
from rl_newton.optimizers.action_space import ActionSpace
from rl_newton.optimizers.controllers import FixedController, OneStepEfficiencyController
from rl_newton.optimizers.newton_cg import NewtonCGConfig, NewtonCGOptimizer
from rl_newton.tasks.quadratics import QuadraticSpec
from rl_newton.types import ControllerAction

spec = QuadraticSpec(kind="ill_conditioned", dimension=50, condition_number=1.0e4)
config = NewtonCGConfig(total_steps=200, cost_budget_ge=BUDGET_GE)
space = ActionSpace(
    name="narrow",
    damping_values=(0.5, 1.0, 2.0),
    cg_budgets=(3, 5, 10, 20),
    step_sizes=(1.0,),
)


def smoke(label, make_controller):
    task = make_task(spec, seed=0)
    optimizer = NewtonCGOptimizer(
        task, make_controller(), config, run_id=f"{label}|notebook", seed=0
    )
    trace = optimizer.run()
    j_e = math.log(trace.initial_loss) - math.log(trace.final_loss)
    print(
        f"{label:<20} steps={trace.n_steps:>3}  object GE={trace.total_cost_ge:>6.1f}  "
        f"search GE={trace.search_cost_ge:>8.1f}  J_E={j_e:>+7.3f}"
    )
    return trace


print(f"{len(space)} actions, {space.n_solve_groups} distinct CG solves per sweep")
print()
smoke("constant k=3", lambda: FixedController(ControllerAction(1.0, 3, 1.0)))
smoke("constant k=10", lambda: FixedController(ControllerAction(1.0, 10, 1.0)))
trace = smoke("one-step greedy", lambda: OneStepEfficiencyController(space))

The one-step controller beats both constants, and it pays for that in search GE. Notice
the pattern the paper reports at scale: state-dependent control is where most of the gain
is, and the search cost is not free.

The loss trajectory below is plotted against **cumulative GE**, not step count. Newton-CG
step cost varies by more than a factor of six across actions, so comparing at equal step
counts would compare runs of unequal cost.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.5, 3.4))
for label, make_controller in [
    ("constant k=3", lambda: FixedController(ControllerAction(1.0, 3, 1.0))),
    ("constant k=10", lambda: FixedController(ControllerAction(1.0, 10, 1.0))),
    ("one-step greedy", lambda: OneStepEfficiencyController(space)),
]:
    task = make_task(spec, seed=0)
    tr = NewtonCGOptimizer(task, make_controller(), config, seed=0).run()
    ax.plot(tr.cumulative_cost_ge(), tr.loss_curve(), marker="o", ms=3, label=label)
ax.set_yscale("log")
ax.set_xlabel("cumulative cost [GE]")
ax.set_ylabel("loss")
ax.set_title("same budget, different resource allocation")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 8. Regenerate the manuscript figures

This runs the same script the manuscript build uses, sourcing the published CSVs instead
of the private raw records. Under the pinned environment of `uv.lock` the output is
byte-for-byte identical to the figures generated from raw. Figure bytes depend on the
matplotlib version, so we do not claim that across different environments.

In [ ]:
out = ROOT / "paper" / "figures"
result = subprocess.run(
    [
        sys.executable,
        str(ROOT / "scripts" / "make_figures.py"),
        "--public-dir",
        str(PUBLIC),
        "--out-dir",
        str(out),
    ],
    capture_output=True,
    text=True,
    cwd=ROOT,
)
print(result.stdout or result.stderr)
assert result.returncode == 0, "figure regeneration failed"

In [ ]:
from IPython.display import Image, display

for name in (
    "figure1_ladder.png",
    "figure2_planning_vs_feedback.png",
    "figure3_model_mismatch.png",
    "figure4_acceptance_ablation.png",
):
    print(name)
    display(Image(filename=str(out / name)))

## 9. What this notebook does not do

- It does not run the full planner suite. That is the dominant cost of the study and is
  documented in `docs/reproduce.md`.
- It does not train a policy. No policy was trained in this work, so nothing here speaks
  to learned-policy performance.
- It does not reimplement any statistic. Every number came from
  `rl_newton.benchmark.metrics` through `rl_newton.reporting`.

To check that the published CSVs agree with the paper across every reported statistic —
not just the ones shown here — run:

```bash
python scripts/verify_public_results.py
pytest tests/ -q
```

### Scope

The confirmatory result is limited to synthetic ill-conditioned SPD quadratics at
`d = 100`. The model-mismatch and acceptance-criterion results are exploratory at
`n = 3` per regime. GE matches oracle calls within a regime, not floating-point
operations across batch sizes. See the manuscript's Limitations section, which also
records twelve protocol deviations.